# Proyecto Final de Machine Learning: Predicción de Tiempos en el Maratón de Boston 2015
**Metodología:** KDD (Knowledge Discovery in Databases) en extenso.
**Enfoque del Modelo:** Regresión.

## 1. Introducción
El presente *notebook* documenta la implementación rigurosa de la metodología KDD sobre el conjunto de datos `marathon_results_2015.csv`, el cual contiene los registros de los corredores del Maratón de Boston del año 2015. 

El **objetivo principal** (Nuestra Tarea de Regresión) es construir un modelo predictivo capaz de estimar el **tiempo oficial de llegada** (variable continua) de un corredor basándonos en sus características demográficas (edad, género) y sus tiempos de paso (ritmo) en los primeros kilómetros de la carrera.

---

## FASE 1: Comprensión del Dominio y Selección de Datos
En esta primera etapa de la metodología KDD, nos enfocamos en recolectar los datos en crudo, cargar las herramientas necesarias y realizar un primer vistazo a la estructura de nuestra información para entender a qué nos enfrentamos.

In [5]:
# ==============================================================================
# CARGA DE LIBRERÍAS
# ==============================================================================
# Si no las tienes instaladas, quita el "#" de la siguiente línea y ejecútala una vez:
install.packages("tidyverse")
install.packages("janitor", repos = "https://cran.rstudio.com/")

library(tidyverse) # Contiene ggplot2 (gráficas), dplyr (manipulación), readr (lectura)
library(janitor)   # Excelente para limpiar los nombres de las columnas automáticamente

# Desactivar notaciones científicas para que los números se vean claros
options(scipen = 999) 

# ==============================================================================
# LECTURA DEL DATASET
# ==============================================================================
# Cargamos los datos
df_maraton <- read_csv("marathon_results_2015.csv")

# Limpiamos los nombres de las columnas (los pasa a minúsculas y quita espacios raros)
df_maraton <- df_maraton %>% clean_names()

# Damos un primer vistazo a la estructura (dimensiones y tipos de datos)
glimpse(df_maraton)

Warning message:
"package 'tidyverse' is in use and will not be installed"
Warning message:
"package 'janitor' is in use and will not be installed"
New names:
• `` -> `...1`
• `` -> `...10`
Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 26598 Columns: 25
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (15): Name, M/F, City, State, Country, Citizen, ...10, 5K, 10K, 20K, Ha...
dbl   (6): ...1, Bib, Age, Overall, Gender, Division
time  (4): 15K, 35K, Pace, Official Time

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 26,598
Columns: 25
$ x1            <dbl> 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16…
$ bib           <dbl> 3, 4, 8, 11, 10, 9, 14, 1, 5, 16, 22, 19, 15, 20, 76, 28…
$ name          <chr> "Desisa, Lelisa", "Tsegay, Yemane Adhane", "Chebet, Wils…
$ age           <dbl> 25, 30, 29, 28, 32, 30, 32, 39, 27, 33, 33, 30, 32, 31, …
$ m_f           <chr> "M", "M", "M", "M", "M", "M", "M", "M", "M", "M", "M", "…
$ city          <chr> "Ambo", "Addis Ababa", "Marakwet", "Eldoret", "Kitale", …
$ state         <chr> NA, NA, NA, NA, NA, NA, "MI", "CA", NA, NA, "OR", "CO", …
$ country       <chr> "ETH", "ETH", "KEN", "KEN", "KEN", "KEN", "USA", "USA", …
$ citizen       <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, …
$ x10           <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, …
$ x5k           <chr> "0:14:43", "0:14:43", "0:14:43", "0:14:43", "0:14:43", "…
$ x10k          <chr> "0:29:43", "0:29:43", "0:29:43", "0:29:44", "0:29:44", "…
$ x15k         

### Interpretación Inicial de los Datos
Al realizar la carga y el primer escrutinio de los datos (mediante `glimpse`), identificamos que el dataset consta de **26,598 observaciones y 25 variables**. 

**Hallazgos y Problemas Identificados (Calidad de Datos):**
1. **Columnas Innecesarias:** Existen variables que no aportan valor predictivo para el rendimiento físico, como el identificador `x1`, `bib` (número de corredor) y `name` (nombre). 
2. **Tipos de Datos Incompatibles:** Las variables de los tiempos parciales (`x5k`, `x10k`, `half`, etc.) y nuestra variable objetivo (`official_time`) fueron detectadas como texto (`<chr>`) o formato temporal (`<time>`). Dado que nuestro enfoque es de **Regresión**, requerimos que estas características sean numéricas (variables continuas).

**Plan de Acción para la Fase 2 (Limpieza y Transformación):**
Procederemos a eliminar las variables redundantes, lidiar con los valores nulos (corredores que no terminaron y tienen un `-` en su tiempo) y transformar todos los registros de tiempo a **minutos totales** (formato numérico decimal) para que el modelo matemático pueda procesarlos correctamente.